In [11]:
# Model 1: no specific constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np


# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_cr = 1000

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

cr_2023 = new_df['cr'].head(N).tolist()


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)  # 严格按公式 B(ak-dj)^{α-1} e^{-B(ak-dj)}
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")

last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
CR = mdl.addVars(N, T_years+1, lb=0, name="CR")

# initialization
for i in range(N):
    mdl.addConstr(last_action[i,0] == 0)
    # mdl.addConstr(CR[i,0] == Y * alpha_init[i]/Delta_P[i]) # calculate initial CR based on alpha_init
    h_2024 = compute_gamma_h(Age_2023[i] + 1, gamma_alpha, gamma_beta)
    mdl.addConstr(CR[i,0] == cr_2023[i] + h_2024) # use given CR 2023 values


# CR演化完整约束（替换步骤3）
for i in range(N):
    for t in range(1, T_years+1):
        # 无行动：CR根据gamma分布自然演化，CR[t] = CR[t-1] + h(t) （h(t)根据年龄计算）
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        # h_cr = compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta)
        h_cr = compute_gamma_h(Age_2023[i] + t, gamma_alpha, gamma_beta)
        mdl.addConstr(CR[i,t] >= CR[i,t-1]+ h_cr - M_cr * (1 - no_action))
        mdl.addConstr(CR[i,t] <= CR[i,t-1] + h_cr + M_cr * (1 - no_action))
        
        # 有行动：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        coper_target = CR[i,t-1] / mu_COPER
        mdl.addConstr(CR[i,t] >= coper_target - M_cr * (1 - x_COPER[i,t-1]))
        mdl.addConstr(CR[i,t] <= coper_target + M_cr * (1 - x_COPER[i,t-1]))

        # MANT维护：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        mant_target = CR[i,t-1] / mu_MANT
        mdl.addConstr(CR[i,t] >= mant_target - M_cr * (1 - x_MANT[i,t-1]))
        mdl.addConstr(CR[i,t] <= mant_target + M_cr * (1 - x_MANT[i,t-1]))


# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (CR[i,t] * Delta_P[i] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# objective
leak_cost = quicksum(
    (CR[i,t] * Delta_P[i] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()



# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        # act_type = "Init"   # 或者 "No action"
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial CR={cr_2023[i]:.1f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), K ={K[i]:.6f}")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                cr_start = cr_2023[i] 
                cr_end = CR[i,0].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t

            else:
                # 使用CR和yearly_leak替换alpha
                cr_start = CR[i,t-1].X
                cr_end = CR[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t-1].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t-1].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                    
            # CR变化百分比
            cr_reduction = (cr_start - cr_end) / cr_start * 100 if cr_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"CR: {cr_start:.3f}→{cr_end:.3f} "
                  f"(-{cr_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print(f"Total leak over entire horizon: {sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years)):.1f} kg")
    print(f"Total COPER actions: {sum(x_COPER[i,t].X for i in range(N) for t in range(T_years)):.0f}")
    print(f"Total MANT actions: {sum(x_MANT[i,t].X for i in range(N) for t in range(T_years)):.0f}")

    


    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")

    print('cr 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {cr_2023[i]:.0f}, csemCur: {new_df['csemCur'].iloc[i]:.1f}")
    non_zero_count = np.count_nonzero(cr_2023)  # 非零个数
    print(f" number of non-zero CR values: {non_zero_count}/{N} ({non_zero_count/N*100:.1f}%)")


    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")
    
else:
    print(f"❌ Optimization status: {mdl.status}")



Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 64512 rows, 23296 columns and 221536 nonzeros
Model fingerprint: 0x9824e1e4
Variable types: 11648 continuous, 11648 integer (11648 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [5e+02, 7e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 1e+06]
Presolve removed 64498 rows and 23289 columns
Presolve time: 0.07s
Presolved: 14 rows, 7 columns, 60 nonzeros
Variable types: 3 continuous, 4 integer (4 binary)
Found heuristic solution: objective 6.429209e+07

Explored 0 nodes (0 simplex iterations) in 0.08 seconds (0.19 work units)
Thread count was 12 (of 12 available processors)

Solution count 1: 6.42921e+

In [5]:

import pandas as pd

# 读取 CSV 文件
df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv')

# 统计 'cr' 列不为 0 的数量
count = len(df[df['cr'] != 0])
print(f"'cr' 列不为 0 的数量: {count}")

'cr' 列不为 0 的数量: 79


In [49]:
# Model 2: budget constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np


# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 3
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()
Total_budget = 27000

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_cr = 1000

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

cr_2023 = new_df['cr'].head(N).tolist()


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)  # 严格按公式 B(ak-dj)^{α-1} e^{-B(ak-dj)}
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")

last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
CR = mdl.addVars(N, T_years+1, lb=0, name="CR")

# initialization
for i in range(N):
    mdl.addConstr(last_action[i,0] == 0)
    # mdl.addConstr(CR[i,0] == Y * alpha_init[i]/Delta_P[i]) # calculate initial CR based on alpha_init
    h_2024 = compute_gamma_h(Age_2023[i] + 1, gamma_alpha, gamma_beta)
    mdl.addConstr(CR[i,0] == cr_2023[i]+h_2024) # use given CR 2023 values


# CR演化完整约束（替换步骤3）
for i in range(N):
    for t in range(1, T_years+1):
        # 无行动：CR根据gamma分布自然演化，CR[t] = CR[t-1] + h(t) （h(t)根据年龄计算）
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        h_cr = compute_gamma_h(Age_2023[i] + t+1, gamma_alpha, gamma_beta)
        mdl.addConstr(CR[i,t] >= CR[i,t-1]+ h_cr - M_cr * (1 - no_action))
        mdl.addConstr(CR[i,t] <= CR[i,t-1] + h_cr + M_cr * (1 - no_action))
        
        # 有行动：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        coper_target = CR[i,t-1] / mu_COPER
        mdl.addConstr(CR[i,t] >= coper_target - M_cr * (1 - x_COPER[i,t-1]))
        mdl.addConstr(CR[i,t] <= coper_target + M_cr * (1 - x_COPER[i,t-1]))

        # MANT维护：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        mant_target = CR[i,t-1] / mu_MANT
        mdl.addConstr(CR[i,t] >= mant_target - M_cr * (1 - x_MANT[i,t-1]))
        mdl.addConstr(CR[i,t] <= mant_target + M_cr * (1 - x_MANT[i,t-1]))


# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (CR[i,t] * Delta_P[i] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！


# Annual action budget constraint
for t in range(T_years):
    yearly_action_cost = quicksum(C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t] for i in range(N))
    mdl.addConstr(yearly_action_cost <= Total_budget, name=f"YearlyBudgetLimit[{t}]")


# objective
leak_cost = quicksum(
    (CR[i,t] * Delta_P[i] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()



# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial CR={cr_2023[i]:.1f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), K ={K[i]:.6f}")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                cr_start = cr_2023[i] 
                cr_end = CR[i,0].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t

            else:
                # 使用CR和yearly_leak替换alpha
                cr_start = CR[i,t-1].X
                cr_end = CR[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t-1].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t-1].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                
            # CR变化百分比
            cr_reduction = (cr_start - cr_end) / cr_start * 100 if cr_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"CR: {cr_start:.3f}→{cr_end:.3f} "
                  f"(-{cr_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")

    print('cr 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {CR[i,0].X}")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")
    
else:
    print(f"❌ Optimization status: {mdl.status}")



Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 881 rows, 312 columns and 3069 nonzeros
Model fingerprint: 0x901773b7
Variable types: 156 continuous, 156 integer (156 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [2e+04, 5e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+06]
Presolve removed 881 rows and 312 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 1: 2.76291e+06 

Optimal solution found (tolerance 5.00e-02)
Best objective 2.762911738472e+06, best bound 2.762911738472e+06, gap 0.

In [50]:
# Model 3: resource constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np


# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 10
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()
L_total = 1

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_cr = 1000

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

cr_2023 = new_df['cr'].head(N).tolist()


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)  # 严格按公式 B(ak-dj)^{α-1} e^{-B(ak-dj)}
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")

last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
CR = mdl.addVars(N, T_years+1, lb=0, name="CR")

# initialization
for i in range(N):
    mdl.addConstr(last_action[i,0] == 0)
    # mdl.addConstr(CR[i,0] == Y * alpha_init[i]/Delta_P[i]) # calculate initial CR based on alpha_init
    h_2024 = compute_gamma_h(Age_2023[i]+1, gamma_alpha, gamma_beta)
    mdl.addConstr(CR[i,0] == cr_2023[i] + h_2024) # use given CR 2023 values


# CR演化完整约束（替换步骤3）
for i in range(N):
    for t in range(1, T_years+1):
        # 无行动：CR根据gamma分布自然演化，CR[t] = CR[t-1] + h(t) （h(t)根据年龄计算）
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        h_cr = compute_gamma_h(Age_2023[i] + t+1, gamma_alpha, gamma_beta)
        mdl.addConstr(CR[i,t] >= CR[i,t-1]+ h_cr - M_cr * (1 - no_action))
        mdl.addConstr(CR[i,t] <= CR[i,t-1] + h_cr + M_cr * (1 - no_action))
        
        # 有行动：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        coper_target = CR[i,t-1] / mu_COPER
        mdl.addConstr(CR[i,t] >= coper_target - M_cr * (1 - x_COPER[i,t-1]))
        mdl.addConstr(CR[i,t] <= coper_target + M_cr * (1 - x_COPER[i,t-1]))

        # MANT维护：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        mant_target = CR[i,t-1] / mu_MANT
        mdl.addConstr(CR[i,t] >= mant_target - M_cr * (1 - x_MANT[i,t-1]))
        mdl.addConstr(CR[i,t] <= mant_target + M_cr * (1 - x_MANT[i,t-1]))


# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (CR[i,t] * Delta_P[i] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！


# Yearly tasks limit
for t in range(T_years):
    mdl.addConstr(quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N)) <= L_total, name=f"Yearly_limit_t{t}")

# objective
leak_cost = quicksum(
    (CR[i,t] * Delta_P[i] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()



# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial CR={cr_2023[i]:.1f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), K ={K[i]:.6f}")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                cr_start = cr_2023[i] 
                cr_end = CR[i,0].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t

            else:
                # 使用CR和yearly_leak替换alpha
                cr_start = CR[i,t-1].X
                cr_end = CR[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t-1].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t-1].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                
            # CR变化百分比
            cr_reduction = (cr_start - cr_end) / cr_start * 100 if cr_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"CR: {cr_start:.3f}→{cr_end:.3f} "
                  f"(-{cr_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")

    print('cr 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {CR[i,0].X}")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")
    
else:
    print(f"❌ Optimization status: {mdl.status}")



Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2897 rows, 1040 columns and 10230 nonzeros
Model fingerprint: 0x262fe94d
Variable types: 520 continuous, 520 integer (520 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [2e+03, 7e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+06]
Presolve removed 2877 rows and 1026 columns
Presolve time: 0.00s
Presolved: 20 rows, 14 columns, 90 nonzeros
Variable types: 6 continuous, 8 integer (8 binary)
Found heuristic solution: objective 9502835.4122

Root relaxation: objective 9.215753e+06, 10 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work


In [57]:
# Model 4:  budget + resource constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np


# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()
Total_budget = 604800
L_total = 16

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_cr = 1000

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

cr_2023 = new_df['cr'].head(N).tolist()


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)  # 严格按公式 B(ak-dj)^{α-1} e^{-B(ak-dj)}
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")

last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
CR = mdl.addVars(N, T_years+1, lb=0, name="CR")

# initialization
for i in range(N):
    mdl.addConstr(last_action[i,0] == 0)
    # mdl.addConstr(CR[i,0] == Y * alpha_init[i]/Delta_P[i]) # calculate initial CR based on alpha_init
    h_2024 = compute_gamma_h(Age_2023[i]+1, gamma_alpha, gamma_beta)
    mdl.addConstr(CR[i,0] == cr_2023[i]+h_2024) # use given CR 2023 values


# CR演化完整约束（替换步骤3）
for i in range(N):
    for t in range(1, T_years+1):
        # 无行动：CR根据gamma分布自然演化，CR[t] = CR[t-1] + h(t) （h(t)根据年龄计算）
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        h_cr = compute_gamma_h(Age_2023[i] + t+1, gamma_alpha, gamma_beta)
        mdl.addConstr(CR[i,t] >= CR[i,t-1]+ h_cr - M_cr * (1 - no_action))
        mdl.addConstr(CR[i,t] <= CR[i,t-1] + h_cr + M_cr * (1 - no_action))
        
        # 有行动：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        coper_target = CR[i,t-1] / mu_COPER
        mdl.addConstr(CR[i,t] >= coper_target - M_cr * (1 - x_COPER[i,t-1]))
        mdl.addConstr(CR[i,t] <= coper_target + M_cr * (1 - x_COPER[i,t-1]))

        # MANT维护：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        mant_target = CR[i,t-1] / mu_MANT
        mdl.addConstr(CR[i,t] >= mant_target - M_cr * (1 - x_MANT[i,t-1]))
        mdl.addConstr(CR[i,t] <= mant_target + M_cr * (1 - x_MANT[i,t-1]))


# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (CR[i,t] * Delta_P[i] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# Annual action budget constraint
for t in range(T_years):
    yearly_action_cost = quicksum(C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t] for i in range(N))
    mdl.addConstr(yearly_action_cost <= Total_budget, name=f"YearlyBudgetLimit[{t}]")


# Yearly tasks limit
for t in range(T_years):
    mdl.addConstr(quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N)) <= L_total, name=f"Yearly_limit_t{t}")

# objective
leak_cost = quicksum(
    (CR[i,t] * Delta_P[i] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()



# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial CR={cr_2023[i]:.1f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), K ={K[i]:.6f}")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                cr_start = cr_2023[i] 
                cr_end = CR[i,0].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                # print(f"  Y{t+1:2d}: Initial CR: {cr_start:.3f}")
                
            else:
                # 使用CR和yearly_leak替换alpha
                cr_start = CR[i,t-1].X
                cr_end = CR[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t-1].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t-1].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                    
            # CR变化百分比
            cr_reduction = (cr_start - cr_end) / cr_start * 100 if cr_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                f"CR: {cr_start:.3f}→{cr_end:.3f} "
                f"(-{cr_reduction:5.1f}%) | "
                f"Leak{leak_kg:>6.3f}kg | "
                f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
    
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print(f"Total leak over entire horizon: {sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years)):.1f} kg")
    print(f"Total COPER actions: {sum(x_COPER[i,t].X for i in range(N) for t in range(T_years)):.0f}")
    print(f"Total MANT actions: {sum(x_MANT[i,t].X for i in range(N) for t in range(T_years)):.0f}")



    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")

    print('cr 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {CR[i,0].X}")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")
    
else:
    print(f"❌ Optimization status: {mdl.status}")



Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 64546 rows, 23296 columns and 236768 nonzeros
Model fingerprint: 0x5e6e132d
Variable types: 11648 continuous, 11648 integer (11648 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [5e+02, 7e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-02, 1e+06]
Presolve removed 64187 rows and 23019 columns
Presolve time: 0.07s
Presolved: 359 rows, 277 columns, 1765 nonzeros
Variable types: 84 continuous, 193 integer (192 binary)
Found heuristic solution: objective 7.583430e+07

Root relaxation: objective 6.946807e+07, 235 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bo

In [9]:
# Model 1: 测试如果我是使用由数据中的alpha算出来的cr作为初始值进行模拟
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np


# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_cr = 1000

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

# cr_2023 = new_df['cr'].head(N).tolist()
cr_2023 = [Y * alpha_init[i] / Delta_P[i] for i in range(N)]  # 直接用alpha计算初始CR


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)  # 严格按公式 B(ak-dj)^{α-1} e^{-B(ak-dj)}
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")

last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
CR = mdl.addVars(N, T_years+1, lb=0, name="CR")

# initialization
for i in range(N):
    mdl.addConstr(last_action[i,0] == 0)
    # mdl.addConstr(CR[i,0] == Y * alpha_init[i]/Delta_P[i]) # calculate initial CR based on alpha_init
    h_2024 = compute_gamma_h(Age_2023[i] + 1, gamma_alpha, gamma_beta)
    mdl.addConstr(CR[i,0] == cr_2023[i] + h_2024) # use given CR 2023 values


# CR演化完整约束（替换步骤3）
for i in range(N):
    for t in range(1, T_years+1):
        # 无行动：CR根据gamma分布自然演化，CR[t] = CR[t-1] + h(t) （h(t)根据年龄计算）
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        h_cr = compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta)
        mdl.addConstr(CR[i,t] >= CR[i,t-1]+ h_cr - M_cr * (1 - no_action))
        mdl.addConstr(CR[i,t] <= CR[i,t-1] + h_cr + M_cr * (1 - no_action))
        
        # 有行动：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        coper_target = CR[i,t-1] / mu_COPER
        mdl.addConstr(CR[i,t] >= coper_target - M_cr * (1 - x_COPER[i,t-1]))
        mdl.addConstr(CR[i,t] <= coper_target + M_cr * (1 - x_COPER[i,t-1]))

        # MANT维护：CR = Y * alpha[t] * K[i] / Delta_P[i] （维护后alpha）
        mant_target = CR[i,t-1] / mu_MANT
        mdl.addConstr(CR[i,t] >= mant_target - M_cr * (1 - x_MANT[i,t-1]))
        mdl.addConstr(CR[i,t] <= mant_target + M_cr * (1 - x_MANT[i,t-1]))


# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (CR[i,t] * Delta_P[i] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# objective
leak_cost = quicksum(
    (CR[i,t] * Delta_P[i] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()



# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        # act_type = "Init"   # 或者 "No action"
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial CR={cr_2023[i]:.1f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), K ={K[i]:.6f}")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                cr_start = cr_2023[i] 
                cr_end = CR[i,0].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t

            else:
                # 使用CR和yearly_leak替换alpha
                cr_start = CR[i,t-1].X
                cr_end = CR[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t-1].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t-1].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                    
            # CR变化百分比
            cr_reduction = (cr_start - cr_end) / cr_start * 100 if cr_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"CR: {cr_start:.3f}→{cr_end:.3f} "
                  f"(-{cr_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print(f"Total leak over entire horizon: {sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years)):.1f} kg")
    print(f"Total COPER actions: {sum(x_COPER[i,t].X for i in range(N) for t in range(T_years)):.0f}")
    print(f"Total MANT actions: {sum(x_MANT[i,t].X for i in range(N) for t in range(T_years)):.0f}")

    


    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")

    print('cr 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {cr_2023[i]:.0f}, csemCur: {new_df['csemCur'].iloc[i]:.1f}")
    non_zero_count = np.count_nonzero(cr_2023)  # 非零个数
    print(f" number of non-zero CR values: {non_zero_count}/{N} ({non_zero_count/N*100:.1f}%)")


    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")
    
else:
    print(f"❌ Optimization status: {mdl.status}")



Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 64512 rows, 23296 columns and 221536 nonzeros
Model fingerprint: 0x091354d9
Variable types: 11648 continuous, 11648 integer (11648 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [5e+02, 7e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e-02, 1e+06]
Presolve removed 64494 rows and 23287 columns
Presolve time: 0.09s
Presolved: 18 rows, 9 columns, 64 nonzeros
Variable types: 4 continuous, 5 integer (5 binary)
Found heuristic solution: objective 6.787550e+07

Explored 0 nodes (0 simplex iterations) in 0.11 seconds (0.21 work units)
Thread count was 12 (of 12 available processors)

Solution count 1: 6.78755e+